In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

np.random.seed(42)

print("Setup complete")

Setup complete


In [2]:
# Cleaned data'ni yuklash (kechagi cleaning natijasi)
df = pd.read_csv('../data/processed/cleaned_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

Dataset shape: (149717, 12)
Columns: ['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'income_missing_flag']

Missing values: 0


In [3]:
# Feature 1a: total_delinquency (oddiy yig'indi)
# Mantiq: barcha kechikishlar birga - umumiy to'lov muammo signal'i

df['total_delinquency'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] 
    + df['NumberOfTime60-89DaysPastDueNotWorse'] 
    + df['NumberOfTimes90DaysLate']
)

# Yangi feature taqsimoti
print("total_delinquency taqsimoti:")
print(df['total_delinquency'].value_counts().sort_index().head(15))
print(f"\nMin:    {df['total_delinquency'].min()}")
print(f"Max:    {df['total_delinquency'].max()}")
print(f"Mean:   {df['total_delinquency'].mean():.2f}")
print(f"Median: {df['total_delinquency'].median()}")

total_delinquency taqsimoti:
total_delinquency
0     119625
1      17242
2       5941
3       2884
4       1595
5        940
6        592
7        380
8        197
9        125
10        73
11        53
12        18
13        19
14        13
Name: count, dtype: int64

Min:    0
Max:    19
Mean:   0.40
Median: 0.0


In [4]:
# Yangi feature target bilan qanday bog'liq?
# total_delinquency qiymati bo'yicha default foizini ko'ramiz

# .groupby() — odamlarni guruhlarga ajratish
# Har guruhda target'ning o'rtachasi = default foizi (chunki target 0 yoki 1)
default_by_delinq = df.groupby('total_delinquency')['SeriousDlqin2yrs'].agg(['mean', 'count'])

# .agg(['mean', 'count']) — bir vaqtda ikkita statistika:
#   mean = default foizi (0-1 oraliqda)
#   count = guruhda nechta odam bor

# Foizga aylantirish
default_by_delinq['mean_pct'] = (default_by_delinq['mean'] * 100).round(2)

print("total_delinquency vs Default rate:")
print(default_by_delinq[['count', 'mean_pct']].head(10))

total_delinquency vs Default rate:
                    count  mean_pct
total_delinquency                  
0                  119625      2.73
1                   17242     12.21
2                    5941     24.07
3                    2884     34.95
4                    1595     42.63
5                     940     50.85
6                     592     58.45
7                     380     60.53
8                     197     59.90
9                     125     67.20


In [5]:
# Feature 2: has_dependents (binary)
# Mantiq: oilali odam vs yolg'iz odam - risk pattern'i farqli

# (df['NumberOfDependents'] > 0) — boolean (True/False)
# .astype(int) — 1/0 ga aylantirish (ML model uchun)
df['has_dependents'] = (df['NumberOfDependents'] > 0).astype(int)

# Taqsimot
print("has_dependents taqsimoti:")
print(df['has_dependents'].value_counts())

# Default rate taqqoslash
print("\nhas_dependents vs Default rate:")
print(df.groupby('has_dependents')['SeriousDlqin2yrs'].agg(['mean', 'count']))

has_dependents taqsimoti:
has_dependents
0    90595
1    59122
Name: count, dtype: int64

has_dependents vs Default rate:
                    mean  count
has_dependents                 
0               0.056935  90595
1               0.079835  59122


In [8]:
df['income_per_dependent']=df['MonthlyIncome']/ (df['NumberOfDependents']+1)


print('income_per_dependent statistika')
print(f"  Min:    {df['income_per_dependent'].min():.2f}")
print(f"  Max:    {df['income_per_dependent'].max():.2f}")
print(f"  Mean:   {df['income_per_dependent'].mean():.2f}")
print(f"  Median: {df['income_per_dependent'].median():.2f}")
df['income_quartile'] = pd.qcut(df['income_per_dependent'], q=4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])

# Endi har guruh uchun default rate
print("\nincome_per_dependent vs Default rate:")
print(df.groupby('income_quartile', observed=True)['SeriousDlqin2yrs'].agg(['mean', 'count']))


income_per_dependent statistika
  Min:    0.00
  Max:    1794060.00
  Mean:   4591.49
  Median: 4000.00

income_per_dependent vs Default rate:
                     mean  count
income_quartile                 
Q1 (low)         0.096073  37430
Q2               0.066844  38298
Q3               0.053668  41608
Q4 (high)        0.045984  32381


In [9]:
# income_per_dependent = 0 bo'lganlar nechta?
zero_income = (df['income_per_dependent'] == 0).sum()
print(f"income_per_dependent = 0 bo'lganlar: {zero_income}")

# Bularning MonthlyIncome qiymati nima?
if zero_income > 0:
    print("\nBu odamlarning MonthlyIncome qiymatlari:")
    print(df[df['income_per_dependent'] == 0]['MonthlyIncome'].value_counts().head())

income_per_dependent = 0 bo'lganlar: 1627

Bu odamlarning MonthlyIncome qiymatlari:
MonthlyIncome
0.0    1627
Name: count, dtype: int64


In [10]:
# MonthlyIncome = 0 odamlarda default rate
zero_income_df = df[df['MonthlyIncome'] == 0]
normal_income_df = df[df['MonthlyIncome'] > 0]

print(f"MonthlyIncome = 0:    {len(zero_income_df)} odam, default rate = {zero_income_df['SeriousDlqin2yrs'].mean()*100:.2f}%")
print(f"MonthlyIncome > 0:    {len(normal_income_df)} odam, default rate = {normal_income_df['SeriousDlqin2yrs'].mean()*100:.2f}%")

MonthlyIncome = 0:    1627 odam, default rate = 3.93%
MonthlyIncome > 0:    148090 odam, default rate = 6.63%


In [11]:
# Feature 4: zero_income_flag
# Mantiq: MonthlyIncome=0 alohida toifa - kutilmagan past default rate
df['zero_income_flag'] = (df['MonthlyIncome'] == 0).astype(int)

# Feature 5: age_group
# Mantiq: yosh kategoriyalari - risk pattern yoshga qarab farq qiladi
# pd.cut() — qiymatlarni belgilangan chegaralarga bo'lish
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 45, 60, 100],          # chegaralar
    labels=['young', 'adult', 'mid_age', 'senior']  # nomlar
)

# Default rate har age_group bo'yicha
print("age_group vs Default rate:")
print(df.groupby('age_group', observed=True)['SeriousDlqin2yrs'].agg(['mean', 'count']))

print("\nzero_income_flag yaratildi")
print(f"Yangi feature'lar soni: {df.shape[1]}")

age_group vs Default rate:
               mean  count
age_group                 
young      0.110964  10607
adult      0.092252  40487
mid_age    0.067648  53586
senior     0.029776  45037

zero_income_flag yaratildi
Yangi feature'lar soni: 19


In [12]:
# Vaqtinchalik feature'ni o'chiramiz - model'ga kerak emas
# (faqat tahlil uchun yaratdik)
df = df.drop(columns=['income_quartile'])

# Final shape
print(f"Final shape: {df.shape}")
print(f"\nUstunlar:")
for col in df.columns:
    print(f"  - {col}")

# Yangi feature-engineered dataset'ni saqlash
output_path = '../data/processed/featured_data.csv'
df.to_csv(output_path, index=False)

import os
file_size_mb = os.path.getsize(output_path) / 1024**2
print(f"\nSaqlandi: {output_path}")
print(f"Fayl o'lchami: {file_size_mb:.2f} MB")

Final shape: (149717, 18)

Ustunlar:
  - SeriousDlqin2yrs
  - RevolvingUtilizationOfUnsecuredLines
  - age
  - NumberOfTime30-59DaysPastDueNotWorse
  - DebtRatio
  - MonthlyIncome
  - NumberOfOpenCreditLinesAndLoans
  - NumberOfTimes90DaysLate
  - NumberRealEstateLoansOrLines
  - NumberOfTime60-89DaysPastDueNotWorse
  - NumberOfDependents
  - income_missing_flag
  - total_delinquency
  - has_dependents
  - income_per_depent
  - income_per_dependent
  - zero_income_flag
  - age_group

Saqlandi: ../data/processed/featured_data.csv
Fayl o'lchami: 11.35 MB


In [28]:
# Hozirgi ustunlar ro'yxati
print(f"Ustunlar soni: {df.shape[1]}")
print("\nUstunlar:")
for col in df.columns:
    print(f"  - {col}")

Ustunlar soni: 12

Ustunlar:
  - SeriousDlqin2yrs
  - RevolvingUtilizationOfUnsecuredLines
  - age
  - NumberOfTime30-59DaysPastDueNotWorse
  - DebtRatio
  - MonthlyIncome
  - NumberOfOpenCreditLinesAndLoans
  - NumberOfTimes90DaysLate
  - NumberRealEstateLoansOrLines
  - NumberOfTime60-89DaysPastDueNotWorse
  - NumberOfDependents
  - income_missing_flag


In [29]:
# Featured data yuklash (kechagi feature engineering natijasi)
df = pd.read_csv('../data/processed/featured_data.csv')

print(f"Shape: {df.shape}")
print(f"\nUstunlar:")
for col in df.columns:
    print(f"  - {col}")

Shape: (149717, 18)

Ustunlar:
  - SeriousDlqin2yrs
  - RevolvingUtilizationOfUnsecuredLines
  - age
  - NumberOfTime30-59DaysPastDueNotWorse
  - DebtRatio
  - MonthlyIncome
  - NumberOfOpenCreditLinesAndLoans
  - NumberOfTimes90DaysLate
  - NumberRealEstateLoansOrLines
  - NumberOfTime60-89DaysPastDueNotWorse
  - NumberOfDependents
  - income_missing_flag
  - total_delinquency
  - has_dependents
  - income_per_depent
  - income_per_dependent
  - zero_income_flag
  - age_group


In [30]:
# Taqqoslash
print("Birinchi 5 qator:")
print(df[['income_per_depent', 'income_per_dependent']].head())

print(f"\nFarqli qatorlar soni: {(df['income_per_depent'] != df['income_per_dependent']).sum()}")
print(f"Jami qatorlar: {len(df)}")

Birinchi 5 qator:
   income_per_depent  income_per_dependent
0             3040.0                3040.0
1             1300.0                1300.0
2             3042.0                3042.0
3             3300.0                3300.0
4            63588.0               63588.0

Farqli qatorlar soni: 0
Jami qatorlar: 149717


In [31]:
# Typo ustunni o'chirish
df = df.drop(columns=['income_per_depent'])

# Tasdiqlash
print(f"Shape: {df.shape}")
print(f"\nQolgan ustunlar:")
for col in df.columns:
    print(f"  - {col}")

# Tozalangan dataset'ni qaytadan saqlash
df.to_csv('../data/processed/featured_data.csv', index=False)
print("\nfeatured_data.csv yangilandi (typo ustun o'chirildi)")

Shape: (149717, 17)

Qolgan ustunlar:
  - SeriousDlqin2yrs
  - RevolvingUtilizationOfUnsecuredLines
  - age
  - NumberOfTime30-59DaysPastDueNotWorse
  - DebtRatio
  - MonthlyIncome
  - NumberOfOpenCreditLinesAndLoans
  - NumberOfTimes90DaysLate
  - NumberRealEstateLoansOrLines
  - NumberOfTime60-89DaysPastDueNotWorse
  - NumberOfDependents
  - income_missing_flag
  - total_delinquency
  - has_dependents
  - income_per_dependent
  - zero_income_flag
  - age_group

featured_data.csv yangilandi (typo ustun o'chirildi)


In [32]:
# pd.cut vs pd.qcut - amaliy taqqoslash

# 1. pd.cut - business chegaralar
df['age_cut'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], 
                       labels=['young', 'adult', 'mid_age', 'senior'])

print("pd.cut (business chegaralar):")
print(df['age_cut'].value_counts().sort_index())
print()

# 2. pd.qcut - teng guruhlar
df['age_qcut'] = pd.qcut(df['age'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

print("pd.qcut (teng guruhlar):")
print(df['age_qcut'].value_counts().sort_index())

# qcut chegaralarini ko'rsatish
print("\nqcut avtomatik tanlagan chegaralar:")
print(pd.qcut(df['age'], q=4).cat.categories)

pd.cut (business chegaralar):
age_cut
young      10607
adult      40487
mid_age    53586
senior     45037
Name: count, dtype: int64

pd.qcut (teng guruhlar):
age_qcut
Q1    38026
Q2    39108
Q3    38352
Q4    34231
Name: count, dtype: int64

qcut avtomatik tanlagan chegaralar:
IntervalIndex([(20.999, 41.0], (41.0, 52.0], (52.0, 63.0], (63.0, 99.0]], dtype='interval[float64, right]')
